# 🌊 B-DRAIN - Flood Risk Prediction Model
**Training Pipeline: Random Forest Classifier**

Notebook ini melatih model ML untuk memprediksi **tingkat risiko banjir** (rendah/sedang/tinggi) per kelurahan di Kota Bekasi berdasarkan fitur spasial, topografi, dan historis banjir.

---
**Dataset:**
- `delineasi-indeks-risiko-banjir.geojson` → label + fitur spasial, RiskType: Banjir Genangan (61 kelurahan)
- `delineasi-indeks-risiko-banjir-bandang.geojson` → label + fitur spasial, RiskType: Banjir Bandang (61 kelurahan)
- `data_banjir.geojson` → fitur historis banjir per kecamatan
- `geojson_kota_bekasi.zip` → **34 layer topografi BIG 1:25.000** (kontur, sungai, pemukiman, sawah, dll)

> Total sampel: **122** (genangan + bandang), fitur diperkaya dari data topografi.

**Fitur Topografi yang Diekstrak:**
| Fitur | Sumber | Deskripsi |
|-------|--------|-----------|
| `elev_mean` | KONTUR + SPOTHEIGHT | Rata-rata elevasi kelurahan (m) |
| `elev_min` | KONTUR + SPOTHEIGHT | Elevasi minimum (area paling rendah) |
| `sungai_density` | SUNGAI_LN | Panjang sungai / luas kelurahan |
| `dist_to_river` | SUNGAI_LN | Jarak centroid ke sungai terdekat |
| `pct_pemukiman` | PEMUKIMAN | % area kedap air (permukiman) |
| `pct_sawah` | AGRISAWAH | % area sawah (menyerap air) |

**Output:**
- `flood_risk_model.pkl` → model terlatih
- `scaler.pkl` → feature scaler
- `model_metadata.json` → info fitur & label

## Step 1: Install & Import Library

In [ ]:
!pip install geopandas scikit-learn xgboost imbalanced-learn matplotlib seaborn joblib -q

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.inspection import permutation_importance
from imblearn.over_sampling import SMOTE

print('✅ Semua library berhasil diimport')

## Step 2: Upload File

**Sebelum upload, zip dulu folder topografi di komputer lokal:**
```
# Di PowerShell (jalankan di folder public/data/):
Compress-Archive -Path geojson_kota_bekasi -DestinationPath geojson_kota_bekasi.zip
```

Upload **4 file** berikut:
1. `delineasi-indeks-risiko-banjir.geojson`
2. `delineasi-indeks-risiko-banjir-bandang.geojson`
3. `data_banjir.geojson`
4. `geojson_kota_bekasi.zip` ← zip dari folder topografi

In [ ]:
from google.colab import files

print('📁 Upload 4 file berikut:')
print('   1. delineasi-indeks-risiko-banjir.geojson')
print('   2. delineasi-indeks-risiko-banjir-bandang.geojson')
print('   3. data_banjir.geojson')
print('   4. geojson_kota_bekasi.zip')
uploaded = files.upload()

required = [
    'delineasi-indeks-risiko-banjir.geojson',
    'delineasi-indeks-risiko-banjir-bandang.geojson',
    'data_banjir.geojson',
    'geojson_kota_bekasi.zip'
]
missing = [f for f in required if f not in uploaded]
if missing:
    print(f'⚠️  File belum terupload: {missing}')
else:
    print(f'✅ Semua file terupload: {list(uploaded.keys())}')

## Step 3: Load & Eksplorasi Data

In [ ]:
# Load GeoJSON risiko banjir genangan
with open('delineasi-indeks-risiko-banjir.geojson', 'r', encoding='utf-8') as f:
    risk_genangan = json.load(f)

# Load GeoJSON risiko banjir bandang
with open('delineasi-indeks-risiko-banjir-bandang.geojson', 'r', encoding='utf-8') as f:
    risk_bandang = json.load(f)

# Load GeoJSON data banjir historis
with open('data_banjir.geojson', 'r', encoding='utf-8') as f:
    banjir_geojson = json.load(f)

print(f'📊 Banjir Genangan : {len(risk_genangan["features"])} kelurahan')
print(f'📊 Banjir Bandang  : {len(risk_bandang["features"])} kelurahan')
print(f'📊 Historis Banjir : {len(banjir_geojson["features"])} kecamatan')
print(f'📊 Total sampel    : {len(risk_genangan["features"]) + len(risk_bandang["features"])} (setelah digabung)')

print('\n🔍 Properti dataset risiko (sample):')
print(list(risk_genangan['features'][0]['properties'].keys()))

## Step 3b: Load Data Topografi (geojson_kota_bekasi)

In [ ]:
import geopandas as gpd
import zipfile, os

# Ekstrak zip topografi
print('📦 Mengekstrak geojson_kota_bekasi.zip...')
with zipfile.ZipFile('geojson_kota_bekasi.zip', 'r') as z:
    z.extractall('topo/')

# Tentukan path (zip bisa menghasilkan subfolder atau langsung file)
if os.path.exists('topo/geojson_kota_bekasi/'):
    TOPO_DIR = 'topo/geojson_kota_bekasi/'
else:
    TOPO_DIR = 'topo/'
print(f'📁 Topo dir: {TOPO_DIR}')
print(f'   Files: {os.listdir(TOPO_DIR)[:5]}...')

# Load layer topografi
gdf_kontur    = gpd.read_file(f'{TOPO_DIR}KONTUR_LN_25K.geojson')
gdf_spot      = gpd.read_file(f'{TOPO_DIR}SPOTHEIGHT_PT_25K.geojson')
gdf_sungai    = gpd.read_file(f'{TOPO_DIR}SUNGAI_LN_25K.geojson')
gdf_pemukiman = gpd.read_file(f'{TOPO_DIR}PEMUKIMAN_AR_25K.geojson')
gdf_sawah     = gpd.read_file(f'{TOPO_DIR}AGRISAWAH_AR_25K.geojson')

# Pastikan CRS sama
for gdf in [gdf_kontur, gdf_spot, gdf_sungai, gdf_pemukiman, gdf_sawah]:
    if gdf.crs is None:
        gdf.set_crs('EPSG:4326', inplace=True)
    else:
        gdf.to_crs('EPSG:4326', inplace=True)

# Konversi kolom numerik
gdf_kontur['VALKNT'] = pd.to_numeric(gdf_kontur['VALKNT'], errors='coerce')
gdf_spot['ELEVAS']   = pd.to_numeric(gdf_spot['ELEVAS'],   errors='coerce')

print('\n✅ Layer topografi berhasil dimuat:')
print(f'   KONTUR     : {len(gdf_kontur):4d} garis kontur (elevasi)')
print(f'   SPOTHEIGHT : {len(gdf_spot):4d} titik ketinggian')
print(f'   SUNGAI_LN  : {len(gdf_sungai):4d} segmen sungai')
print(f'   PEMUKIMAN  : {len(gdf_pemukiman):4d} area permukiman')
print(f'   SAWAH      : {len(gdf_sawah):4d} area sawah')

In [ ]:
# Buat GeoDataFrame kelurahan unik dari data genangan (geometri identik dengan bandang)
features_kel = []
seen_kel = set()
for feat in risk_genangan['features']:
    name = feat['properties'].get('Kelurahan', feat['properties'].get('NAMOBJ', ''))
    if name and name not in seen_kel:
        seen_kel.add(name)
        features_kel.append(feat)

gdf_kel = gpd.GeoDataFrame.from_features(features_kel, crs='EPSG:4326')
gdf_kel['kel_key']   = [f['properties'].get('Kelurahan', f['properties'].get('NAMOBJ','')) for f in features_kel]
gdf_kel['area_deg2'] = gdf_kel['SHAPE_Area'].astype(float)
print(f'📐 Total kelurahan polygon: {len(gdf_kel)}')

# Ekstraksi fitur topografi per kelurahan
print('⏳ Mengekstrak fitur topografi per kelurahan (sekitar 30-60 detik)...')
topo_records = []

for idx, kel_row in gdf_kel.iterrows():
    kel_geom  = kel_row.geometry
    kel_name  = kel_row['kel_key']
    kel_area  = float(kel_row['area_deg2']) if float(kel_row['area_deg2']) > 0 else 1e-10
    centroid  = kel_geom.centroid

    # ── ELEVASI: dari SPOTHEIGHT (titik dalam polygon) ──
    pts_in = gdf_spot[gdf_spot.geometry.within(kel_geom)]
    if len(pts_in) > 0:
        vals      = pts_in['ELEVAS'].dropna()
        elev_mean = float(vals.mean()) if len(vals) > 0 else np.nan
        elev_min  = float(vals.min())  if len(vals) > 0 else np.nan
    else:
        # Fallback: garis kontur yang berinterseksi dengan polygon
        kon_in = gdf_kontur[gdf_kontur.geometry.intersects(kel_geom)]
        if len(kon_in) > 0:
            vals      = kon_in['VALKNT'].dropna()
            elev_mean = float(vals.mean()) if len(vals) > 0 else np.nan
            elev_min  = float(vals.min())  if len(vals) > 0 else np.nan
        else:
            elev_mean = np.nan
            elev_min  = np.nan

    # ── SUNGAI: kepadatan & jarak ke sungai ──
    sun_in = gdf_sungai[gdf_sungai.geometry.intersects(kel_geom)]
    if len(sun_in) > 0:
        total_len     = float(sun_in.geometry.length.sum())
        sungai_density = total_len / kel_area
        dist_to_river  = float(sun_in.geometry.distance(centroid).min())
    else:
        sungai_density = 0.0
        dist_to_river  = float(gdf_sungai.geometry.distance(centroid).min())

    # ── PEMUKIMAN: rasio area kedap air ──
    pem_in = gdf_pemukiman[gdf_pemukiman.geometry.intersects(kel_geom)]
    if len(pem_in) > 0:
        pem_area      = float(pem_in.geometry.intersection(kel_geom).area.sum())
        pct_pemukiman = pem_area / kel_area
    else:
        pct_pemukiman = 0.0

    # ── SAWAH: rasio area penyerap air ──
    saw_in = gdf_sawah[gdf_sawah.geometry.intersects(kel_geom)]
    if len(saw_in) > 0:
        saw_area  = float(saw_in.geometry.intersection(kel_geom).area.sum())
        pct_sawah = saw_area / kel_area
    else:
        pct_sawah = 0.0

    topo_records.append({
        'kelurahan':      kel_name,
        'elev_mean':      elev_mean,
        'elev_min':       elev_min,
        'sungai_density': sungai_density,
        'dist_to_river':  dist_to_river,
        'pct_pemukiman':  min(pct_pemukiman, 1.0),  # cap 100%
        'pct_sawah':      min(pct_sawah, 1.0),
    })

df_topo = pd.DataFrame(topo_records)

# Fill NaN dengan median
for col in ['elev_mean', 'elev_min']:
    df_topo[col] = df_topo[col].fillna(df_topo[col].median())

print(f'\n✅ Ekstraksi topo selesai: {len(df_topo)} kelurahan')
print(f'\n📊 Statistik fitur topografi:')
print(df_topo.describe().round(4))
df_topo.head()

In [ ]:
def extract_risk_features(geojson_data, is_bandang=0):
    """Ekstrak fitur dari GeoJSON delineasi risiko banjir."""
    records = []
    for feature in geojson_data['features']:
        props = feature['properties']
        geom = feature['geometry']

        if geom and geom['type'] == 'Polygon':
            coords = np.array(geom['coordinates'][0])
            centroid_lon = float(np.mean(coords[:, 0]))
            centroid_lat = float(np.mean(coords[:, 1]))
            perimeter = float(np.sum(np.sqrt(np.diff(coords[:, 0])**2 + np.diff(coords[:, 1])**2)))
        elif geom and geom['type'] == 'MultiPolygon':
            all_coords = []
            for poly in geom['coordinates']:
                all_coords.extend(poly[0])
            coords = np.array(all_coords)
            centroid_lon = float(np.mean(coords[:, 0]))
            centroid_lat = float(np.mean(coords[:, 1]))
            perimeter = float(props.get('SHAPE_Leng', 0))
        else:
            centroid_lon, centroid_lat, perimeter = 0.0, 0.0, 0.0

        area = float(props.get('SHAPE_Area', 0) or 0)
        leng = float(props.get('SHAPE_Leng', 1) or 1)

        record = {
            'kelurahan':      str(props.get('Kelurahan', props.get('NAMOBJ', ''))),
            'kecamatan':      str(props.get('Kecamatan', props.get('WADMKC', ''))),
            'shape_area':     area,
            'shape_leng':     leng,
            'centroid_lon':   centroid_lon,
            'centroid_lat':   centroid_lat,
            # Compactness ratio (bentuk bulat=1, memanjang<1)
            'compactness':    float((4 * np.pi * area) / (leng ** 2)) if leng > 0 else 0,
            # Fitur pembeda tipe banjir: 0=genangan, 1=bandang
            'is_bandang':     is_bandang,
            'kelas':          str(props.get('Kelas', '')).lower().strip(),
        }
        records.append(record)
    return records

# Ekstrak kedua dataset
records_genangan = extract_risk_features(risk_genangan, is_bandang=0)
records_bandang  = extract_risk_features(risk_bandang,  is_bandang=1)

# Gabungkan
df_risk = pd.DataFrame(records_genangan + records_bandang)

print(f'✅ Total sampel gabungan: {len(df_risk)}')
print(f'   - Banjir Genangan : {sum(df_risk["is_bandang"] == 0)}')
print(f'   - Banjir Bandang  : {sum(df_risk["is_bandang"] == 1)}')
print(f'\n📊 Distribusi Kelas Risiko (gabungan):')
print(df_risk['kelas'].value_counts())
df_risk.head()

In [ ]:
# Ekstrak fitur dari data_banjir.geojson (data historis per kecamatan)
banjir_records = []

for feature in banjir_geojson['features']:
    props = feature['properties']
    record = {
        'kecamatan': str(props.get('kecamatan', props.get('nama_kecamatan', ''))).upper().strip(),
        'jumlah_titik_banjir': int(props.get('jumlah_titik_banjir', 0) or 0),
        'masa_tanggap_darurat_hari': int(props.get('masa_tanggap_darurat_hari', 0) or 0),
        'severity_historis': str(props.get('severity', 'low'))
    }
    banjir_records.append(record)

df_banjir = pd.DataFrame(banjir_records)

# Enkode severity historis ke angka
severity_map = {'low': 1, 'medium': 2, 'high': 3}
df_banjir['severity_score'] = df_banjir['severity_historis'].map(severity_map).fillna(1)

print(f'✅ Extracted {len(df_banjir)} kecamatan records')
df_banjir.head()

In [ ]:
# Normalisasi nama kecamatan untuk join
df_risk['kecamatan_upper'] = df_risk['kecamatan'].str.upper().str.strip()
df_banjir['kecamatan_upper'] = df_banjir['kecamatan'].str.upper().str.strip()

# Merge 1: risk + banjir historis (by kecamatan)
df_merged = df_risk.merge(
    df_banjir[['kecamatan_upper', 'jumlah_titik_banjir', 'masa_tanggap_darurat_hari', 'severity_score']],
    on='kecamatan_upper',
    how='left'
)

# Merge 2: + fitur topografi (by kelurahan)
df_merged = df_merged.merge(
    df_topo[['kelurahan', 'elev_mean', 'elev_min', 'sungai_density', 'dist_to_river', 'pct_pemukiman', 'pct_sawah']],
    on='kelurahan',
    how='left'
)

# Fill NaN
fill_median_cols = [
    'jumlah_titik_banjir', 'masa_tanggap_darurat_hari', 'severity_score',
    'elev_mean', 'elev_min', 'sungai_density', 'dist_to_river',
    'pct_pemukiman', 'pct_sawah'
]
for col in fill_median_cols:
    df_merged[col] = df_merged[col].fillna(df_merged[col].median())

print(f'✅ Dataset setelah merge: {len(df_merged)} rows')
print(f'Missing values:\n{df_merged.isnull().sum()[df_merged.isnull().sum() > 0]}')
print('\nSample kolom topo per kelurahan:')
print(df_merged[['kelurahan', 'elev_mean', 'elev_min', 'sungai_density', 'dist_to_river', 'pct_pemukiman', 'pct_sawah', 'kelas']].head(8))

## Step 4: Visualisasi Data (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('B-DRAIN - EDA (Genangan + Bandang + Topografi)', fontsize=14, fontweight='bold')

# 1. Distribusi kelas per tipe banjir
kelas_order = ['rendah', 'sedang', 'tinggi']
x = np.arange(len(kelas_order)); width = 0.35
gen_counts = [len(df_merged[(df_merged['kelas']==k)&(df_merged['is_bandang']==0)]) for k in kelas_order]
ban_counts  = [len(df_merged[(df_merged['kelas']==k)&(df_merged['is_bandang']==1)]) for k in kelas_order]
axes[0,0].bar(x - width/2, gen_counts, width, label='Genangan', color='#1e40af', alpha=0.8)
axes[0,0].bar(x + width/2, ban_counts, width, label='Bandang',  color='#991b1b', alpha=0.8)
axes[0,0].set_xticks(x); axes[0,0].set_xticklabels(kelas_order)
axes[0,0].set_title('Distribusi Kelas per Tipe Banjir')
axes[0,0].set_ylabel('Jumlah Kelurahan'); axes[0,0].legend()

# 2. Elevasi mean per kelas (FITUR BARU)
kelas_color = {'rendah':'#065f46', 'sedang':'#c2410c', 'tinggi':'#991b1b'}
for kelas, color in kelas_color.items():
    subset = df_merged[df_merged['kelas'] == kelas]['elev_mean']
    axes[0,1].scatter([kelas]*len(subset), subset, c=color, alpha=0.5, s=25)
df_merged.boxplot(column='elev_mean', by='kelas', ax=axes[0,1], 
                  boxprops=dict(color='#374151'), positions=[0,1,2])
axes[0,1].set_title('Elevasi Rata-rata per Kelas ← FITUR KUNCI')
axes[0,1].set_xlabel('Kelas'); axes[0,1].set_ylabel('Elevasi (m)')

# 3. Kepadatan sungai per kelas (FITUR BARU)
df_merged.boxplot(column='sungai_density', by='kelas', ax=axes[0,2],
                  boxprops=dict(color='#1e40af'))
axes[0,2].set_title('Kepadatan Sungai per Kelas')
axes[0,2].set_xlabel('Kelas'); axes[0,2].set_ylabel('Sungai / Luas')

# 4. % Pemukiman vs Elevasi (scatter berwarna kelas)
for kelas, color in kelas_color.items():
    s = df_merged[df_merged['kelas'] == kelas]
    axes[1,0].scatter(s['pct_pemukiman']*100, s['elev_mean'],
                      c=color, label=kelas, alpha=0.7, s=35)
axes[1,0].set_title('% Pemukiman vs Elevasi')
axes[1,0].set_xlabel('% Pemukiman (lahan kedap air)')
axes[1,0].set_ylabel('Elevasi Rata-rata (m)'); axes[1,0].legend()

# 5. Jarak ke sungai per kelas
df_merged.boxplot(column='dist_to_river', by='kelas', ax=axes[1,1],
                  boxprops=dict(color='#065f46'))
axes[1,1].set_title('Jarak ke Sungai per Kelas')
axes[1,1].set_xlabel('Kelas'); axes[1,1].set_ylabel('Jarak (derajat)')

# 6. Korelasi heatmap semua fitur
feature_cols_corr = [
    'shape_area', 'compactness', 'is_bandang',
    'elev_mean', 'elev_min', 'sungai_density',
    'dist_to_river', 'pct_pemukiman', 'pct_sawah',
    'jumlah_titik_banjir', 'severity_score'
]
# Encode kelas ke numeric untuk heatmap
kelas_enc = df_merged['kelas'].map({'rendah':0,'sedang':1,'tinggi':2})
corr_df = df_merged[feature_cols_corr].copy()
corr_df['kelas'] = kelas_enc
corr = corr_df.corr()
sns.heatmap(corr, ax=axes[1,2], annot=True, fmt='.2f', cmap='RdYlGn',
            linewidths=0.5, annot_kws={'size': 6}, vmin=-1, vmax=1)
axes[1,2].set_title('Korelasi Fitur vs Kelas')

plt.tight_layout()
plt.savefig('eda_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ EDA selesai')

## Step 5: Persiapan Data untuk Training

In [ ]:
# Definisi fitur dan label
FEATURE_COLS = [
    # ── Fitur Spasial Kelurahan ──
    'shape_area',                # Luas wilayah kelurahan
    'shape_leng',                # Keliling wilayah
    'compactness',               # Rasio kompaktness bentuk wilayah
    'centroid_lat',              # Posisi lintang
    'centroid_lon',              # Posisi bujur
    'is_bandang',                # Tipe banjir: 0=genangan, 1=bandang
    # ── Fitur Topografi (dari geojson_kota_bekasi) ──
    'elev_mean',                 # Rata-rata elevasi dalam kelurahan (m)
    'elev_min',                  # Elevasi minimum (titik terendah)
    'sungai_density',            # Panjang sungai / luas kelurahan
    'dist_to_river',             # Jarak centroid ke sungai terdekat (derajat)
    'pct_pemukiman',             # Proporsi area permukiman (lahan kedap air)
    'pct_sawah',                 # Proporsi area sawah (penyerap air)
    # ── Fitur Historis Banjir ──
    'jumlah_titik_banjir',       # Titik banjir historis di kecamatan
    'masa_tanggap_darurat_hari', # Durasi tanggap darurat historis
    'severity_score'             # Severity historis (1=low, 2=medium, 3=high)
]
LABEL_COL = 'kelas'

# Filter hanya data dengan kelas valid
df_clean = df_merged[df_merged[LABEL_COL].isin(['rendah', 'sedang', 'tinggi'])].copy()
print(f'✅ Data siap training: {len(df_clean)} rows ({len(FEATURE_COLS)} fitur)')
print(f'   - Banjir Genangan : {sum(df_clean["is_bandang"] == 0)}')
print(f'   - Banjir Bandang  : {sum(df_clean["is_bandang"] == 1)}')
print(f'\nFitur ({len(FEATURE_COLS)} total):')
for i, col in enumerate(FEATURE_COLS, 1):
    print(f'   {i:2d}. {col}')
print(f'\nDistribusi kelas:')
print(df_clean[LABEL_COL].value_counts())

X = df_clean[FEATURE_COLS].values
y_raw = df_clean[LABEL_COL].values

# Encode label: rendah=0, sedang=1, tinggi=2
le = LabelEncoder()
le.classes_ = np.array(['rendah', 'sedang', 'tinggi'])
y = le.transform(y_raw)
print(f'\nLabel encoding: {dict(zip(le.classes_, range(len(le.classes_))))}')

In [ ]:
# Normalisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data: 80% train, 20% test (stratified agar proporsi kelas terjaga)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train size : {X_train.shape[0]} | Test size: {X_test.shape[0]}')
print(f'Train kelas: {dict(zip(le.classes_, np.bincount(y_train)))}')
print(f'Test  kelas: {dict(zip(le.classes_, np.bincount(y_test)))}')

# Handle class imbalance dengan SMOTE
# Strategy: upsample rendah & sedang mendekati tinggi
min_count = np.min(np.bincount(y_train))
try:
    # Oversample semua kelas minoritas ke 80% jumlah kelas mayoritas
    target_count = int(np.max(np.bincount(y_train)) * 0.8)
    sampling_strategy = {
        i: max(np.bincount(y_train)[i], target_count)
        for i in range(len(le.classes_))
    }
    smote = SMOTE(
        random_state=42,
        k_neighbors=min(3, min_count - 1),
        sampling_strategy=sampling_strategy
    )
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
    print(f'\n✅ Setelah SMOTE:')
    print(f'   Train size: {X_train_resampled.shape[0]}')
    print(f'   Distribusi: {dict(zip(le.classes_, np.bincount(y_train_resampled)))}')
except Exception as e:
    print(f'⚠️ SMOTE gagal ({e}), pakai data asli + class_weight')
    X_train_resampled, y_train_resampled = X_train, y_train

## Step 6: Training Model

In [ ]:
# ============================================
# Model 1: Random Forest Classifier
# ============================================
print('🌲 Training Random Forest...')

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_resampled, y_train_resampled)

# Evaluasi
y_pred_rf = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, y_pred_rf)

print(f'\n📊 Random Forest Accuracy: {rf_accuracy:.4f} ({rf_accuracy*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
# ============================================
# Model 2: Gradient Boosting Classifier
# ============================================
print('🚀 Training Gradient Boosting...')

gb_model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)

gb_model.fit(X_train_resampled, y_train_resampled)

y_pred_gb = gb_model.predict(X_test)
gb_accuracy = accuracy_score(y_test, y_pred_gb)

print(f'\n📊 Gradient Boosting Accuracy: {gb_accuracy:.4f} ({gb_accuracy*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_gb, target_names=le.classes_))

In [ ]:
from xgboost import XGBClassifier

# ============================================
# Model 3: XGBoost — Fix class imbalance sedang
# ============================================
print('⚡ Training XGBoost...')

# Hitung class weight manual (makin langka makin besar bobotnya)
class_counts = np.bincount(y_train_resampled)
total        = len(y_train_resampled)
# weight[i] = total / (n_classes * count[i])
sample_weights = np.array([
    total / (len(class_counts) * class_counts[yi]) for yi in y_train_resampled
])

xgb_model = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=42,
    verbosity=0
)

xgb_model.fit(X_train_resampled, y_train_resampled, sample_weight=sample_weights)

y_pred_xgb  = xgb_model.predict(X_test)
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)

print(f'\n📊 XGBoost Accuracy: {xgb_accuracy:.4f} ({xgb_accuracy*100:.2f}%)')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))

# ── Cek apakah sedang membaik ──
from sklearn.metrics import f1_score
f1_sedang_gb  = f1_score(y_test, y_pred_gb,  labels=[1], average='micro')
f1_sedang_xgb = f1_score(y_test, y_pred_xgb, labels=[1], average='micro')
print(f'📈 F1 kelas SEDANG — GB: {f1_sedang_gb:.3f} | XGB: {f1_sedang_xgb:.3f}')

In [ ]:
# ============================================
# Cross Validation 5-Fold — 3 Model
# ============================================
print('🔁 Cross Validation (5-Fold Stratified)...\n')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_cv_scores  = cross_val_score(rf_model,  X_scaled, y, cv=cv, scoring='accuracy')
gb_cv_scores  = cross_val_score(gb_model,  X_scaled, y, cv=cv, scoring='accuracy')
xgb_cv_scores = cross_val_score(xgb_model, X_scaled, y, cv=cv, scoring='accuracy')

# CV macro-F1 (lebih adil untuk imbalanced)
from sklearn.metrics import make_scorer
f1_macro_scorer = make_scorer(f1_score, average='macro')
rf_f1_cv  = cross_val_score(rf_model,  X_scaled, y, cv=cv, scoring=f1_macro_scorer)
gb_f1_cv  = cross_val_score(gb_model,  X_scaled, y, cv=cv, scoring=f1_macro_scorer)
xgb_f1_cv = cross_val_score(xgb_model, X_scaled, y, cv=cv, scoring=f1_macro_scorer)

print(f'{"Model":<22} {"CV Accuracy":>15}  {"CV Macro-F1":>15}')
print('─' * 55)
for name, acc, f1 in [
    ('🌲 Random Forest',    rf_cv_scores,  rf_f1_cv),
    ('🚀 Gradient Boosting', gb_cv_scores,  gb_f1_cv),
    ('⚡ XGBoost',           xgb_cv_scores, xgb_f1_cv),
]:
    print(f'{name:<22}  {acc.mean():.4f} ± {acc.std():.3f}   {f1.mean():.4f} ± {f1.std():.3f}')

# Pilih model terbaik berdasarkan macro-F1 (lebih fair untuk imbalanced)
candidates = [
    ('Random Forest',    rf_model,  rf_cv_scores,  rf_f1_cv),
    ('Gradient Boosting', gb_model, gb_cv_scores,  gb_f1_cv),
    ('XGBoost',          xgb_model, xgb_cv_scores, xgb_f1_cv),
]
best_name, best_model, best_cv_scores, best_f1_cv = max(candidates, key=lambda x: x[3].mean())
best_cv_score = best_cv_scores.mean()
best_f1_score = best_f1_cv.mean()

print(f'\n🏆 Model terbaik : {best_name}')
print(f'   CV Accuracy   : {best_cv_score*100:.2f}%')
print(f'   CV Macro-F1   : {best_f1_score:.4f}')
print(f'\n💡 Macro-F1 digunakan sebagai metric utama karena class imbalance')

## Step 7: Evaluasi Visual

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle(f'Model Evaluation ─ 3 Kandidat (★ terbaik: {best_name})', fontsize=13, fontweight='bold')

all_models = [
    ('🌲 Random Forest',    rf_model,  y_pred_rf),
    ('🚀 Gradient Boosting', gb_model, y_pred_gb),
    ('⚡ XGBoost',           xgb_model, y_pred_xgb),
]

# Row 1: Confusion Matrix per model
for i, (mname, mobj, ypred) in enumerate(all_models):
    ax = fig.add_subplot(2, 3, i+1)
    cm = confusion_matrix(y_test, ypred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=le.classes_, yticklabels=le.classes_)
    acc = accuracy_score(y_test, ypred)
    f1  = f1_score(y_test, ypred, average='macro')
    star = ' ★' if mname.split()[-1] == best_name.split()[-1] else ''
    ax.set_title(f'{mname}{star}\nAcc={acc:.2f} | F1={f1:.2f}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

# Row 2 kiri: Feature Importance (model terbaik)
ax4 = fig.add_subplot(2, 3, 4)
if hasattr(best_model, 'feature_importances_'):
    imp = best_model.feature_importances_
    feat_df = pd.DataFrame({'feature': FEATURE_COLS, 'importance': imp})
    feat_df = feat_df.sort_values('importance', ascending=True)
    colors_bar = ['#991b1b' if imp > feat_df['importance'].quantile(0.75)
                  else '#1e40af' for imp in feat_df['importance']]
    ax4.barh(feat_df['feature'], feat_df['importance'], color=colors_bar)
    ax4.set_title(f'Feature Importance ({best_name})\n🔴=top fitur')
    ax4.set_xlabel('Importance')

# Row 2 tengah: CV Accuracy + Macro-F1 comparison
ax5 = fig.add_subplot(2, 3, 5)
m_labels = ['RF', 'GB', 'XGB']
acc_means = [rf_cv_scores.mean(), gb_cv_scores.mean(), xgb_cv_scores.mean()]
f1_means  = [rf_f1_cv.mean(),    gb_f1_cv.mean(),    xgb_f1_cv.mean()]
x_pos = np.arange(len(m_labels)); w = 0.35
ax5.bar(x_pos - w/2, acc_means, w, label='CV Accuracy', color='#1e40af', alpha=0.8)
ax5.bar(x_pos + w/2, f1_means,  w, label='CV Macro-F1', color='#991b1b', alpha=0.8)
ax5.set_xticks(x_pos); ax5.set_xticklabels(m_labels)
ax5.set_ylim(0, 1.1); ax5.set_title('CV Accuracy vs Macro-F1\n(Macro-F1 lebih adil untuk imbalanced)')
ax5.legend()
for j, (a, f) in enumerate(zip(acc_means, f1_means)):
    ax5.text(j - w/2, a + 0.02, f'{a:.3f}', ha='center', fontsize=8)
    ax5.text(j + w/2, f + 0.02, f'{f:.3f}', ha='center', fontsize=8)

# Row 2 kanan: F1 per kelas (model terbaik)
ax6 = fig.add_subplot(2, 3, 6)
from sklearn.metrics import classification_report as cr
cr_dict = cr(y_test, best_model.predict(X_test), target_names=le.classes_, output_dict=True)
kelas_names = le.classes_.tolist()
f1_per_class   = [cr_dict[k]['f1-score']  for k in kelas_names]
prec_per_class = [cr_dict[k]['precision'] for k in kelas_names]
rec_per_class  = [cr_dict[k]['recall']    for k in kelas_names]
x3 = np.arange(len(kelas_names)); w3 = 0.25
ax6.bar(x3 - w3,    prec_per_class, w3, label='Precision', color='#065f46', alpha=0.8)
ax6.bar(x3,         rec_per_class,  w3, label='Recall',    color='#c2410c', alpha=0.8)
ax6.bar(x3 + w3,    f1_per_class,   w3, label='F1-score',  color='#1e40af', alpha=0.8)
ax6.set_xticks(x3); ax6.set_xticklabels(kelas_names)
ax6.set_ylim(0, 1.2); ax6.set_title(f'Precision/Recall/F1 per Kelas\n({best_name})')
ax6.legend(fontsize=8)
for j, f1v in enumerate(f1_per_class):
    ax6.text(j + w3, f1v + 0.03, f'{f1v:.2f}', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\n✅ Evaluasi selesai. Model terpilih: {best_name}')

## Step 8: Simpan Model

In [ ]:
joblib.dump(best_model,   'flood_risk_model.pkl')
joblib.dump(scaler,       'scaler.pkl')
joblib.dump(le,           'label_encoder.pkl')

# Simpan metadata untuk API
metadata = {
    'model_name':          best_name,
    'cv_accuracy':         float(best_cv_score),
    'cv_macro_f1':         float(best_f1_score),
    'feature_columns':     FEATURE_COLS,
    'label_classes':       le.classes_.tolist(),
    'label_mapping':       {cls: int(i) for i, cls in enumerate(le.classes_)},
    'training_samples':    int(len(df_clean)),
    'class_distribution':  df_clean['kelas'].value_counts().to_dict(),
    'all_models_cv': {
        'Random Forest':    {'accuracy': float(rf_cv_scores.mean()),  'macro_f1': float(rf_f1_cv.mean())},
        'Gradient Boosting':{'accuracy': float(gb_cv_scores.mean()),  'macro_f1': float(gb_f1_cv.mean())},
        'XGBoost':          {'accuracy': float(xgb_cv_scores.mean()), 'macro_f1': float(xgb_f1_cv.mean())},
    }
}

with open('model_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print('✅ Model tersimpan:')
print('   - flood_risk_model.pkl')
print('   - scaler.pkl')
print('   - label_encoder.pkl')
print('   - model_metadata.json')
print(f'\n📊 Model Terpilih : {best_name}')
print(f'   CV Accuracy   : {best_cv_score*100:.2f}%')
print(f'   CV Macro-F1   : {best_f1_score:.4f}')
print(f'\n📊 Perbandingan semua model:')
for mname, scores in metadata['all_models_cv'].items():
    star = ' ← terpilih' if mname == best_name else ''
    print(f'   {mname:<22}: Acc={scores["accuracy"]:.3f} | Macro-F1={scores["macro_f1"]:.3f}{star}')

In [ ]:
# Download semua file hasil training
from google.colab import files

for filename in ['flood_risk_model.pkl', 'scaler.pkl', 'label_encoder.pkl', 'model_metadata.json']:
    files.download(filename)
    print(f'⬇️ Downloading {filename}...')

print('\n✅ Semua file berhasil didownload!')
print('📁 Simpan file-file tersebut ke folder: Project/ml/models/')

## Step 9: Test Prediksi Manual

In [ ]:
# Test prediksi untuk 1 kelurahan
# Ganti nilai-nilai ini sesuai data kelurahan yang ingin diprediksi

test_input = {
    # Fitur spasial
    'shape_area':     0.0005,
    'shape_leng':     0.15,
    'compactness':    0.28,
    'centroid_lat':   -6.25,
    'centroid_lon':   107.0,
    'is_bandang':     0,          # 0=genangan, 1=bandang
    # Fitur topografi ← dari geojson_kota_bekasi
    'elev_mean':      8.0,        # Elevasi rata-rata (m) — Bekasi ~5-40m
    'elev_min':       3.0,        # Elevasi minimum (m)
    'sungai_density': 0.08,       # Kepadatan sungai
    'dist_to_river':  0.005,      # Jarak centroid ke sungai (derajat ~500m)
    'pct_pemukiman':  0.65,       # 65% area adalah permukiman
    'pct_sawah':      0.05,       # 5% area adalah sawah
    # Fitur historis
    'jumlah_titik_banjir':        28,
    'masa_tanggap_darurat_hari':  14,
    'severity_score': 3
}

X_in = np.array([[test_input[f] for f in FEATURE_COLS]])
X_in_scaled = scaler.transform(X_in)

pred   = best_model.predict(X_in_scaled)[0]
probs  = best_model.predict_proba(X_in_scaled)[0]
label  = le.inverse_transform([pred])[0]
tipe   = 'Banjir Bandang' if test_input['is_bandang'] == 1 else 'Banjir Genangan'

print(f'🔮 Hasil Prediksi ({tipe}):')
print(f'   Kelas Risiko: {label.upper()}')
print(f'   Probabilitas:')
for kelas, prob in zip(le.classes_, probs):
    bar = '█' * int(prob * 25)
    print(f'     {kelas:8s}: {bar:25s} {prob*100:.1f}%')

# Bandingkan: area rendah (elevasi tinggi) vs area rawan (elevasi rendah)
print('\n─── Perbandingan Skenario ───')
for scenario in [
    {'label': 'Area Dataran Tinggi (elev 30m, jauh sungai)', 'elev_mean': 30, 'elev_min': 25, 'dist_to_river': 0.05, 'pct_pemukiman': 0.3},
    {'label': 'Area Dataran Rendah (elev 5m, dekat sungai)', 'elev_mean': 5,  'elev_min': 2,  'dist_to_river': 0.002, 'pct_pemukiman': 0.8},
]:
    sc = test_input.copy()
    sc.update(scenario)
    X_sc = scaler.transform(np.array([[sc[f] for f in FEATURE_COLS]]))
    pred_sc = le.inverse_transform(best_model.predict(X_sc))[0]
    prob_sc = max(best_model.predict_proba(X_sc)[0])
    print(f'  {scenario["label"]:50s} → {pred_sc.upper()} ({prob_sc*100:.0f}%)')